# Fase 3 — Semana 2: pipeline de preprocesamiento en clases

**Grupo 4 · MCDI500 · Encuesta Nacional de Salud 2016-2017**

En la Sumativa 1 dejamos listo un conjunto de 5.511 personas para
estudiar cómo se asocian edad, sexo, escolaridad, ingreso y zona con
cinco indicadores de riesgo cardiovascular: hipertensión, diabetes,
colesterol alto, índice de masa corporal y actividad física. Ese
preprocesamiento vivía en funciones sueltas dentro de un notebook.

En esta entrega reescribimos esos mismos pasos como clases que
comparten una interfaz común. El criterio de éxito es concreto: el
pipeline con clases debe entregar exactamente el mismo conjunto que
guardamos en la Fase 2.

| Sección | Contenido |
|---|---|
| 1 | Configuración y carga del conjunto elegible F1-F2 |
| 2 | Pipeline de preprocesamiento en clases |
| 3 | Verificación contra el resultado de la Fase 2 |
| 4 | Validación: caso normal, casos límite y excepciones |
| 5 | Eficiencia: tiempo y memoria |
| 6 | Patrón de diseño Strategy aplicado a la imputación |
| 7 | Arquitectura y conclusiones |

## 1. Configuración y carga del conjunto elegible F1-F2

Partimos del archivo filtrado por ponderador en la Fase 2, antes de
la limpieza. Así las clases tienen que reproducir todo el
preprocesamiento, y el resultado se puede comparar con el conjunto
final guardado en esa fase. Las columnas se agrupan según su rol en
el estudio: predictoras sociodemográficas, indicadores de riesgo y
variables del diseño muestral.

In [4]:
RUTA_DATOS = "data/processed/ens_variables_f1f2.xlsx"
COLUMNA_ID = "IdEncuesta"

# Predictoras sociodemográficas
COLUMNAS_PREDICTORAS_CONTINUAS = ["Edad", "anos_estudio_MINSAL_1", "as27"]
COLUMNAS_PREDICTORAS_NOMINALES = ["Sexo", "Zona"]
COLUMNAS_PREDICTORAS_ORDINALES = ["as28"]

# Indicadores de riesgo cardiovascular (se analizan por separado)
COLUMNAS_RESULTADO_BINARIAS = ["HTA"]
COLUMNAS_RESULTADO_NOMINALES = ["di3", "dis2"]
COLUMNAS_RESULTADO_ORDINALES = ["GPAQ"]
COLUMNAS_RESULTADO_CONTINUAS = ["IMC"]

# Diseño muestral: se conservan sin transformar
COLUMNAS_DISENO_MUESTRAL = ["Fexp_F1F2p_Corr", "Conglomerado", "Estrato"]

SEMILLA = 2026

COLUMNAS_ESPERADAS = (
    [COLUMNA_ID]
    + COLUMNAS_PREDICTORAS_CONTINUAS + COLUMNAS_PREDICTORAS_NOMINALES
    + COLUMNAS_PREDICTORAS_ORDINALES + COLUMNAS_RESULTADO_BINARIAS
    + COLUMNAS_RESULTADO_NOMINALES + COLUMNAS_RESULTADO_ORDINALES
    + COLUMNAS_RESULTADO_CONTINUAS + COLUMNAS_DISENO_MUESTRAL
)
print("Columnas declaradas:", len(COLUMNAS_ESPERADAS))

Columnas declaradas: 15


In [55]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# La raíz se busca aquí porque es la que permite importar src/;
# por eso no puede venir desde el propio src/carga.py.
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), None)
if RAIZ is None:
    raise FileNotFoundError("No se encontró la raíz del repositorio (.git).")
sys.path.append(str(RAIZ / "src"))

from carga import cargar_conjunto, perfilar
from transformador import Transformador

np.random.seed(SEMILLA)
print("pandas", pd.__version__, "· NumPy", np.__version__, "· semilla", SEMILLA)

pandas 3.0.5 · NumPy 2.5.3 · semilla 2026


In [6]:
datos = cargar_conjunto(RAIZ / RUTA_DATOS, COLUMNAS_ESPERADAS)
perfil = perfilar(datos)
perfil[perfil["nulos"] > 0]

Conjunto leído: 5520 filas x 16 columnas
Columnas no usadas en el análisis: ['FechaInicioF1']


,tipo,nulos,pct_nulos
HTA,float64,9,0.16
IMC,float64,37,0.67
anos_estudio_MINSAL_1,float64,47,0.85
GPAQ,float64,196,3.55
as27,float64,995,18.03


El conjunto tiene 5.520 personas. Los nulos se concentran en `as27`
(995), `GPAQ` (196), `anos_estudio_MINSAL_1` (47), `IMC` (37) y `HTA`
(9). La no respuesta de `as28` no aparece aquí porque está codificada
como -9999: el pipeline debe convertirla en nulo antes de imputar.

## 2. Pipeline de preprocesamiento en clases

Cada paso de limpieza de la Fase 2 se reescribe como una subclase de
`Transformador` (`src/transformador.py`). La clase base fija el orden
de uso: `ajustar()` calcula los parámetros con el conjunto de
referencia y `transformar()` los aplica sobre una copia, sin volver a
calcularlos. Cada subclase solo define qué calcula (`aprender()`) y
cómo lo usa (`aplicar()`). Los pasos concretos de imputación,
codificación y escalamiento se incorporan en las subsecciones
siguientes.

## 2.1 Preparar los datos antes de imputar

Antes de rellenar cualquier hueco hay que resolver dos cosas que en la
Fase 2 se hicieron a mano.

La primera son los códigos de no respuesta. En `as28` (tramo de ingreso
del hogar) hay 818 personas con el valor -9999, que pandas trata como
un ingreso más. `MarcadorNoRespuesta` los convierte en nulos, pero
antes deja anotado en `as28_no_responde` quién no respondió: no
declarar el ingreso puede tener un significado propio y no conviene
perderlo.

La segunda son las 9 personas sin diagnóstico de hipertensión (`HTA`).
Un diagnóstico no se puede estimar sin afirmar algo que nadie declaró,
así que `EliminadorFilasNulas` las quita en lugar de rellenarlas.

Las dos clases heredan de `Transformador`: solo definen qué calculan y
cómo lo aplican. El orden de uso (ajustar antes de transformar) lo
controla la clase base.

In [7]:
from imputadores import (
    MarcadorNoRespuesta, EliminadorFilasNulas, ImputadorFlexible,
    PorMedia, PorMediana, PorModa, PorMedianaDeTramo, comparar_estrategias,
)

marcador = MarcadorNoRespuesta("as28")
marcado = marcador.ajustar_transformar(datos)
print(marcador)
print("Personas con código -9999 en as28 (antes):", int((datos["as28"] == -9999).sum()))
print("Nulos en as28 (después):", int(marcado["as28"].isna().sum()))
print("Marcadas en as28_no_responde:", int(marcado["as28_no_responde"].sum()))

eliminador = EliminadorFilasNulas("HTA")
sin_hta = eliminador.ajustar_transformar(marcado)
print(eliminador)
print(f"Filas: {len(marcado)} -> {len(sin_hta)} ({len(marcado) - len(sin_hta)} eliminadas por HTA nulo)")
print("as28_no_responde tras eliminar:", int(sin_hta["as28_no_responde"].sum()))

MarcadorNoRespuesta(as28) [ajustado]
Personas con código -9999 en as28 (antes): 818
Nulos en as28 (después): 818
Marcadas en as28_no_responde: 818
EliminadorFilasNulas(HTA) [ajustado]
Filas: 5520 -> 5511 (9 eliminadas por HTA nulo)
as28_no_responde tras eliminar: 816


Se marcaron 818 personas, pero después de quitar las 9 filas sin `HTA`
quedan 816: dos de esas nueve también tenían -9999 en `as28`.

El orden importa. La bandera se crea antes de borrar los códigos, y las
filas se eliminan antes de imputar, porque en la Fase 2 las medianas se
calcularon sobre las 5.511 personas restantes. Si se imputara antes, los
valores de relleno cambiarían un poco y el resultado ya no coincidiría.

## 2.2 Imputación: una clase y varias formas de rellenar

`ImputadorFlexible` recibe la columna y una estrategia, y no sabe
rellenar por sí sola: le pide a la estrategia que calcule con qué y que
lo aplique. Las decisiones son las de la Fase 2:

| Columna | Estrategia | Por qué |
|---|---|---|
| `IMC`, `anos_estudio_MINSAL_1` | `PorMediana` | Distribuciones asimétricas |
| `GPAQ` | `PorModa` | Es ordinal (3,6 % de nulos): la moda conserva una categoría real |
| `as27` | `PorMedianaDeTramo` sobre `as28` | Solo se imputan las 177 personas que sí declararon su tramo; las 816 que no respondieron ninguno de los dos datos quedan sin valor |

In [8]:
pasos_imputacion = [
    ImputadorFlexible("IMC", PorMediana()),
    ImputadorFlexible("anos_estudio_MINSAL_1", PorMediana()),
    ImputadorFlexible("as27", PorMedianaDeTramo("as28")),
    ImputadorFlexible("GPAQ", PorModa()),
]

imputado = sin_hta
for paso in pasos_imputacion:
    nulos_antes = int(imputado[paso.columna].isna().sum())
    imputado = paso.ajustar_transformar(imputado)   # misma llamada para todos
    nulos_despues = int(imputado[paso.columna].isna().sum())
    print(f"{paso.nombre:<40} nulos: {nulos_antes:>4} -> {nulos_despues}")

n_imputadas = int(imputado["as27_imputado"].sum())
n_sin_dato = int(imputado["as27"].isna().sum())
print("as27 imputadas por tramo:", n_imputadas)
print("as27 que quedan sin dato:", n_sin_dato)

assert n_imputadas == 177 and n_sin_dato == 816
assert imputado[["IMC", "anos_estudio_MINSAL_1", "GPAQ"]].isna().sum().sum() == 0
assert len(imputado) == len(sin_hta), "La imputación no debe cambiar el número de filas"
print("Verificado: los conteos coinciden con los de la Fase 2.")

ImputadorFlexible(IMC)                   nulos:   37 -> 0
ImputadorFlexible(anos_estudio_MINSAL_1) nulos:   47 -> 0
ImputadorFlexible(as27)                  nulos:  993 -> 816
ImputadorFlexible(GPAQ)                  nulos:  196 -> 0
as27 imputadas por tramo: 177
as27 que quedan sin dato: 816
Verificado: los conteos coinciden con los de la Fase 2.


Herencia, polimorfismo y encapsulamiento se ven así en este código:

- **Herencia:** los tres tipos de paso parten de `Transformador`; ninguno
  reescribió el manejo del estado.
- **Polimorfismo:** todos los pasos se usan con la misma llamada,
  `ajustar_transformar`, y `ImputadorFlexible` pide `calcular` y
  `rellenar` a la estrategia sin saber si es media, mediana, moda o
  por tramo.
- **Encapsulamiento:** lo aprendido queda guardado en `_parametros`; desde
  fuera solo se obtiene una copia, y usar un paso sin ajustarlo produce
  un error.

### 2.3 Comprobación del encapsulamiento

Las tres pruebas siguientes muestran el estado protegido. Primero se
transforma sin haber ajustado, algo que debe fallar (el error se captura
a propósito). Luego se usa el orden correcto, aprendiendo con una parte
de las personas y aplicando a la otra. Por último se intenta reemplazar
desde fuera lo que el paso aprendió.

In [9]:
# 1) Transformar antes de ajustar: debe fallar, con un mensaje claro
paso_nuevo = ImputadorFlexible("IMC", PorMediana())
try:
    paso_nuevo.transformar(datos)
except RuntimeError as error:
    print("RuntimeError:", error)

# 2) Orden correcto: se aprende en entrenamiento y se aplica a prueba
entrenamiento = sin_hta.sample(frac=0.8, random_state=SEMILLA)
prueba = sin_hta.drop(index=entrenamiento.index)

paso = ImputadorFlexible("IMC", PorMediana()).ajustar(entrenamiento)
prueba_lista = paso.transformar(prueba)

print("\nMediana aprendida en entrenamiento   :", round(paso.parametros["valor"], 2))
print("Mediana propia del conjunto de prueba:", round(prueba["IMC"].median(), 2))
print("Nulos de IMC en prueba tras imputar  :", int(prueba_lista["IMC"].isna().sum()))

# 3) Lo aprendido no se puede reemplazar desde fuera
try:
    paso.parametros = {"valor": -1}
except AttributeError as error:
    print("\nAttributeError:", error)

RuntimeError: ImputadorFlexible(IMC): falta llamar a ajustar().

Mediana aprendida en entrenamiento   : 28.3
Mediana propia del conjunto de prueba: 28.37
Nulos de IMC en prueba tras imputar  : 0

AttributeError: property 'parametros' of 'ImputadorFlexible' object has no setter


Sin la comprobación previa, aplicar un paso antes de ajustarlo fallaría
de forma confusa o, peor, seguiría de largo sin avisar. Aquí el problema
aparece de inmediato y con una explicación.

Las dos medianas son parecidas, pero no idénticas. Lo importante es cuál
se usa: a la prueba se le aplica la del entrenamiento, así sus propios
datos no influyen en el relleno, aunque la diferencia sea pequeña.

## 3. Codificación de variables categóricas

Como parte del pipeline de preprocesamiento orientado a objetos, se incorpora la clase `CodificadorOneHot`, que hereda de la clase base `Transformador`.

Su objetivo es transformar las variables categóricas seleccionadas de la ENS en columnas binarias mediante **One-Hot Encoding**, manteniendo nombres semánticos asociados a las categorías originales.

Las variables consideradas son:

- `Sexo`
- `Zona`
- `di3`
- `dis2`

La implementación mantiene la misma interfaz utilizada por los demás componentes del pipeline: `ajustar()`, `transformar()` y `ajustar_transformar()`.

In [63]:
from src.transformadores import (
    CodificadorOneHot,
    EscaladorEstandar,
)

### 3.1 Categorías utilizadas

La codificación utiliza las categorías definidas para las variables seleccionadas de la ENS.

Además de realizar la transformación, `CodificadorOneHot` valida durante el ajuste que los códigos observados correspondan a categorías conocidas. De esta manera, un código no contemplado genera una excepción en lugar de ser procesado silenciosamente.

In [57]:
variables_categoricas = ["Sexo", "Zona", "di3", "dis2"]

for columna in variables_categoricas:
    print(f"\n--- {columna} ---")
    print(df[columna].value_counts(dropna=False).sort_index())


--- Sexo ---
Sexo
1    2315
2    3918
Name: count, dtype: int64

--- Zona ---
Zona
1    5242
2     991
Name: count, dtype: int64

--- di3 ---
di3
1     886
2    5294
3      53
Name: count, dtype: int64

--- dis2 ---
dis2
1     834
2     693
3    4394
4     312
Name: count, dtype: int64


### 3.2 Aplicación de `CodificadorOneHot`

Cada variable categórica se transforma utilizando una instancia independiente de `CodificadorOneHot`.

Durante `ajustar()`, el transformador valida los códigos presentes y almacena las categorías correspondientes. Posteriormente, `transformar()` genera una columna binaria por categoría y elimina la variable categórica original.

El proceso se aplica secuencialmente para mantener la lógica común definida por la clase base `Transformador`.

In [59]:
df_codificado = df.copy()

codificadores = {}

for columna in variables_categoricas:
    codificador = CodificadorOneHot(columna)

    df_codificado = codificador.ajustar_transformar(
        df_codificado
    )

    codificadores[columna] = codificador

print("Dimensiones antes de codificar:", df.shape)
print("Dimensiones después de codificar:", df_codificado.shape)

Dimensiones antes de codificar: (6233, 1173)
Dimensiones después de codificar: (6233, 1180)


In [60]:
columnas_onehot = [
    columna
    for columna in df_codificado.columns
    if columna.startswith(("Sexo_", "Zona_", "di3_", "dis2_"))
]

print("Columnas generadas:")
for columna in columnas_onehot:
    print("-", columna)

Columnas generadas:
- Sexo_Hombre
- Sexo_Mujer
- Zona_Urbano
- Zona_Rural
- di3_Si
- di3_No
- di3_No_recuerda
- dis2_Si_una_vez
- dis2_Si_mas_de_una_vez
- dis2_Nunca
- dis2_No_recuerda


In [61]:
print("Valores únicos por columna:\n")

for columna in columnas_onehot:
    print(
        columna,
        sorted(df_codificado[columna].unique())
    )

Valores únicos por columna:

Sexo_Hombre [np.int64(0), np.int64(1)]
Sexo_Mujer [np.int64(0), np.int64(1)]
Zona_Urbano [np.int64(0), np.int64(1)]
Zona_Rural [np.int64(0), np.int64(1)]
di3_Si [np.int64(0), np.int64(1)]
di3_No [np.int64(0), np.int64(1)]
di3_No_recuerda [np.int64(0), np.int64(1)]
dis2_Si_una_vez [np.int64(0), np.int64(1)]
dis2_Si_mas_de_una_vez [np.int64(0), np.int64(1)]
dis2_Nunca [np.int64(0), np.int64(1)]
dis2_No_recuerda [np.int64(0), np.int64(1)]


### 3.3 Validación de la codificación

La transformación generó correctamente las variables binarias asociadas a las categorías de `Sexo`, `Zona`, `di3` y `dis2`.

Las columnas resultantes contienen exclusivamente valores `0` y `1`, mientras que las variables categóricas originales son retiradas del conjunto transformado.

La implementación permite además conservar la semántica de las categorías mediante nombres descriptivos, evitando utilizar únicamente los códigos numéricos originales de la ENS.

Las validaciones unitarias y los casos de excepción de este componente se encuentran implementados separadamente en `tests/test_transformadores.py`.

## 4. Escalamiento de variables numéricas

Como siguiente etapa del pipeline se incorpora `EscaladorEstandar`, una subclase de `Transformador` destinada a estandarizar variables numéricas.

Para una observación \(x\), la transformación aplicada corresponde a:

\[
z = \frac{x-\mu}{\sigma}
\]

donde:

- \(x\) es el valor original;
- \(\mu\) es la media calculada durante el ajuste;
- \(\sigma\) es la desviación estándar calculada durante el ajuste.

La separación entre `ajustar()` y `transformar()` permite almacenar los parámetros aprendidos y reutilizarlos posteriormente sobre nuevos datos.

### 4.1 Verificación previa al escalamiento

Antes de aplicar una transformación numérica se verifica que las variables seleccionadas no contengan valores faltantes.

Esta comprobación es relevante porque el escalamiento corresponde a una etapa posterior al tratamiento de valores nulos dentro del pipeline de preprocesamiento.

In [64]:
variables_numericas = ["Edad", "IMC", "as27"]

print("Nulos en df:")
print(df[variables_numericas].isna().sum())

print("\nTipos de datos:")
print(df[variables_numericas].dtypes)

Nulos en df:
Edad       0
IMC      750
as27    1133
dtype: int64

Tipos de datos:
Edad      int64
IMC     float64
as27    float64
dtype: object


In [65]:
variables_numericas = ["Edad", "IMC", "as27"]

candidatos = [
    nombre for nombre, valor in globals().items()
    if isinstance(valor, pd.DataFrame)
]

print("DataFrames disponibles:")
for nombre in candidatos:
    dataframe = globals()[nombre]

    if all(col in dataframe.columns for col in variables_numericas):
        print(
            nombre,
            "->",
            dataframe.shape,
            "| nulos:",
            dataframe[variables_numericas].isna().sum().to_dict()
        )

DataFrames disponibles:
datos -> (5520, 16) | nulos: {'Edad': 0, 'IMC': 37, 'as27': 995}
marcado -> (5520, 17) | nulos: {'Edad': 0, 'IMC': 37, 'as27': 995}
sin_hta -> (5511, 17) | nulos: {'Edad': 0, 'IMC': 37, 'as27': 993}
imputado -> (5511, 18) | nulos: {'Edad': 0, 'IMC': 0, 'as27': 816}
entrenamiento -> (4409, 17) | nulos: {'Edad': 0, 'IMC': 33, 'as27': 812}
prueba -> (1102, 17) | nulos: {'Edad': 0, 'IMC': 4, 'as27': 181}
prueba_lista -> (1102, 17) | nulos: {'Edad': 0, 'IMC': 0, 'as27': 181}
resultado -> (5511, 18) | nulos: {'Edad': 0, 'IMC': 37, 'as27': 816}
df -> (6233, 1173) | nulos: {'Edad': 0, 'IMC': 750, 'as27': 1133}
df_codificado -> (6233, 1180) | nulos: {'Edad': 0, 'IMC': 750, 'as27': 1133}
df_numerico -> (6233, 3) | nulos: {'Edad': 0, 'IMC': 750, 'as27': 1133}
df_numerico_prueba -> (4495, 3) | nulos: {'Edad': 0, 'IMC': 0, 'as27': 0}
df_escalado -> (4495, 3) | nulos: {'Edad': 0, 'IMC': 0, 'as27': 0}
_35 -> (5, 3) | nulos: {'Edad': 0, 'IMC': 0, 'as27': 0}


In [66]:
print("Estado de as27 después de la etapa de imputación:")
print("Registros totales:", len(imputado))
print("Valores disponibles:", imputado["as27"].notna().sum())
print("Valores nulos:", imputado["as27"].isna().sum())

print("\nResumen estadístico de as27:")
print(imputado["as27"].describe())

Estado de as27 después de la etapa de imputación:
Registros totales: 5511
Valores disponibles: 4695
Valores nulos: 816

Resumen estadístico de as27:
count    4.695000e+03
mean     4.689790e+05
std      6.202431e+05
min      2.000000e+04
25%      2.000000e+05
50%      3.000000e+05
75%      5.000000e+05
max      1.600000e+07
Name: as27, dtype: float64


In [67]:
columnas_ingreso = [
    columna for columna in imputado.columns
    if "as27" in columna.lower() or "as28" in columna.lower()
]

print("Columnas relacionadas con ingreso:")
print(columnas_ingreso)

Columnas relacionadas con ingreso:
['as27', 'as28', 'as28_no_responde', 'as27_imputado']


### 4.2 Aplicación de `EscaladorEstandar`

El escalamiento se aplica después de la etapa de imputación.

`Edad` e `IMC` pueden escalarse sobre la totalidad de los registros disponibles. En el caso de `as27`, se conservan los valores faltantes definidos por la estrategia de imputación de la Fase 2.

La estrategia `PorMedianaDeTramo` imputa únicamente los casos en que existe información del tramo de ingreso (`as28`). Los participantes que no entregaron información suficiente mantienen `as27` como valor faltante, evitando introducir ingresos artificiales.

Por lo tanto, el escalamiento no modifica la política de tratamiento de valores faltantes definida previamente.

In [70]:
df_escalado = imputado.copy()

variables_numericas = ["Edad", "IMC", "as27"]

escaladores = {}

for columna in variables_numericas:
    escalador = EscaladorEstandar(columna)

    df_escalado = escalador.ajustar_transformar(
        df_escalado
    )

    escaladores[columna] = escalador

print("Dimensiones:", df_escalado.shape)

print("\nNulos después del escalamiento:")
print(df_escalado[variables_numericas].isna().sum())

Dimensiones: (5511, 18)

Nulos después del escalamiento:
Edad      0
IMC       0
as27    816
dtype: int64


### 4.3 Validación estadística del escalamiento

Para verificar el funcionamiento de `EscaladorEstandar`, se comprueba que las variables transformadas presenten una media cercana a 0 y una desviación estándar cercana a 1.

En `as27`, las estadísticas se calculan sobre los valores disponibles. Los 816 valores faltantes definidos por la estrategia de imputación anterior se conservan y no intervienen en el cálculo.

In [71]:
for columna in variables_numericas:
    media = df_escalado[columna].mean()
    desviacion = df_escalado[columna].std(ddof=0)

    print(f"\n{columna}")
    print("Media:", round(media, 10))
    print("Desviación estándar:", round(desviacion, 10))


Edad
Media: 0.0
Desviación estándar: 1.0

IMC
Media: -0.0
Desviación estándar: 1.0

as27
Media: -0.0
Desviación estándar: 1.0


### 4.4 Resultado del escalamiento

Las tres variables presentan una media aproximadamente igual a 0 y una desviación estándar igual a 1 después de la transformación, confirmando el funcionamiento esperado de `EscaladorEstandar`.

En `as27` se mantienen los 816 valores faltantes provenientes de la etapa anterior. El escalador transforma únicamente los valores disponibles y no modifica la estrategia de imputación definida previamente.

Esta separación de responsabilidades permite que cada componente del pipeline mantenga una función específica: los imputadores gestionan los valores faltantes y el escalador realiza exclusivamente la transformación numérica.

## 5. Evaluación de eficiencia

Además de validar el funcionamiento de los transformadores, se evalúa su eficiencia computacional mediante comparaciones con implementaciones de referencia.

Se consideran dos dimensiones:

- **Tiempo de ejecución:** permite comparar el costo temporal de cada implementación.
- **Memoria pico:** permite estimar la memoria utilizada durante la ejecución.

Las mediciones se realizan mediante las funciones reutilizables definidas en `src/medicion.py`.

Se comparan:

1. `EscaladorEstandar` frente a `StandardScaler` de scikit-learn.
2. `CodificadorOneHot` frente a `pandas.get_dummies`.

Además de las mediciones empíricas, se analiza la complejidad temporal y espacial de las soluciones.

In [76]:
from src.medicion import (
    medir_tiempo,
    medir_memoria,
)

In [77]:
def escalar_propio(df, columna):
    escalador = EscaladorEstandar(columna)

    return escalador.ajustar_transformar(
        df[[columna]].copy()
    )


def escalar_sklearn(df, columna):
    escalador = StandardScaler()

    resultado = df[[columna]].copy()

    resultado[columna] = escalador.fit_transform(
        resultado[[columna]]
    ).ravel()

    return resultado

### 5.1 Equivalencia de resultados

Antes de comparar eficiencia, se verifica que ambas implementaciones produzcan resultados numéricamente equivalentes.

Esta comprobación evita comparar el rendimiento de algoritmos que realizan transformaciones diferentes.

In [75]:
datos_edad = imputado[["Edad"]].copy()

resultado_propio = escalar_propio(
    datos_edad,
    "Edad"
)

resultado_sklearn = escalar_sklearn(
    datos_edad,
    "Edad"
)

np.testing.assert_allclose(
    resultado_propio["Edad"].to_numpy(),
    resultado_sklearn["Edad"].to_numpy(),
    rtol=1e-10,
    atol=1e-10,
)

print("Resultados equivalentes: EscaladorEstandar == StandardScaler")

Resultados equivalentes: EscaladorEstandar == StandardScaler


### 5.2 Medición de tiempo y memoria del escalamiento

Una vez comprobada la equivalencia entre ambas implementaciones, se evalúa su comportamiento para diferentes tamaños de entrada.

Se utilizan subconjuntos de 100, 500, 1.000, 2.500, 5.000 y 5.511 observaciones. Para cada tamaño se mide:

- tiempo de ejecución;
- memoria pico utilizada.

El objetivo es observar cómo evoluciona el costo computacional al aumentar el número de registros y complementar las mediciones empíricas con un análisis de complejidad.

In [78]:
tamanos = [100, 500, 1000, 2500, 5000, len(imputado)]

resultados_escalamiento = []

for n in tamanos:
    muestra = imputado[["Edad"]].iloc[:n].copy()

    tiempo_propio, _ = medir_tiempo(
        escalar_propio,
        muestra,
        "Edad"
    )

    tiempo_sklearn, _ = medir_tiempo(
        escalar_sklearn,
        muestra,
        "Edad"
    )

    memoria_propia, _ = medir_memoria(
        escalar_propio,
        muestra,
        "Edad"
    )

    memoria_sklearn, _ = medir_memoria(
        escalar_sklearn,
        muestra,
        "Edad"
    )

    resultados_escalamiento.append({
        "n": n,
        "tiempo_propio_s": tiempo_propio,
        "tiempo_sklearn_s": tiempo_sklearn,
        "memoria_propia_kb": memoria_propia / 1024,
        "memoria_sklearn_kb": memoria_sklearn / 1024,
    })

resultados_escalamiento = pd.DataFrame(
    resultados_escalamiento
)

resultados_escalamiento

,n,tiempo_propio_s,tiempo_sklearn_s,memoria_propia_kb,memoria_sklearn_kb
0,100,0.000497,0.001799,10.477539,12.486328
1,500,0.000470,0.001707,23.032227,22.238281
2,1000,0.000496,0.001698,38.657227,34.490234
3,2500,0.000474,0.001698,85.532227,71.111328
4,5000,0.000484,0.001733,163.657227,132.101562
5,5511,0.000469,0.001670,179.625977,144.577148


### 5.3 Análisis de eficiencia del escalamiento

Las mediciones muestran que `EscaladorEstandar` presentó menores tiempos de ejecución que `StandardScaler` en todos los tamaños evaluados.

Para 5.511 observaciones, la implementación propia registró aproximadamente 0,000469 s, mientras que `StandardScaler` registró 0,001670 s, por lo que la implementación propia fue aproximadamente 3,56 veces más rápida en este experimento.

En memoria se observa un comportamiento diferente. Para 5.511 observaciones, `EscaladorEstandar` alcanzó aproximadamente 179,63 KB de memoria pico, mientras que `StandardScaler` registró 144,58 KB. Por lo tanto, la implementación de scikit-learn presentó un menor consumo de memoria en los tamaños mayores.

Desde el punto de vista de complejidad, ambas implementaciones requieren recorrer las observaciones para calcular los parámetros estadísticos y aplicar la transformación. Por esta razón, su complejidad temporal respecto del número de observaciones es O(n).

La aparente estabilidad de los tiempos experimentales no implica una complejidad O(1), ya que para el volumen evaluado los tiempos son muy pequeños y los costos fijos de ejecución tienen una influencia importante sobre la medición.

Considerando la equivalencia numérica previamente verificada, los menores tiempos observados y su integración directa con la arquitectura orientada a objetos del pipeline, se mantiene `EscaladorEstandar` como implementación del proyecto. El mayor consumo de memoria observado se considera un compromiso aceptable para el volumen actual de datos.

In [79]:
def codificar_propio(df, columna):
    codificador = CodificadorOneHot(columna)

    return codificador.ajustar_transformar(
        df[[columna]].copy()
    )


def codificar_pandas(df, columna):
    resultado = pd.get_dummies(
        df[[columna]],
        columns=[columna],
        prefix=columna,
        dtype=int
    )

    if columna == "Sexo":
        resultado = resultado.rename(
            columns={
                "Sexo_1": "Sexo_Hombre",
                "Sexo_2": "Sexo_Mujer",
            }
        )

    return resultado

In [80]:
datos_sexo = imputado[["Sexo"]].copy()

resultado_propio_ohe = codificar_propio(
    datos_sexo,
    "Sexo"
)

resultado_pandas_ohe = codificar_pandas(
    datos_sexo,
    "Sexo"
)

pd.testing.assert_frame_equal(
    resultado_propio_ohe,
    resultado_pandas_ohe
)

print(
    "Resultados equivalentes: "
    "CodificadorOneHot == pandas.get_dummies"
)

Resultados equivalentes: CodificadorOneHot == pandas.get_dummies


### 5.4 Evaluación de eficiencia de `CodificadorOneHot`

Para evaluar el segundo transformador desarrollado, se compara `CodificadorOneHot` con `pandas.get_dummies`.

Primero se verifica que ambas implementaciones generen una representación binaria equivalente. Posteriormente se comparan sus tiempos de ejecución y memoria pico para distintos tamaños de entrada.

### 5.5 Medición de tiempo y memoria de la codificación

Una vez comprobada la equivalencia entre `CodificadorOneHot` y `pandas.get_dummies`, se evalúa el rendimiento de ambas implementaciones utilizando diferentes tamaños de entrada.

Se utilizan subconjuntos de 100, 500, 1.000, 2.500, 5.000 y 5.511 observaciones, manteniendo el mismo criterio utilizado en la evaluación del escalamiento.

Para cada tamaño se registra el tiempo de ejecución y la memoria pico utilizada.

In [81]:
tamanos = [100, 500, 1000, 2500, 5000, len(imputado)]

resultados_codificacion = []

for n in tamanos:
    muestra = imputado[["Sexo"]].iloc[:n].copy()

    tiempo_propio, _ = medir_tiempo(
        codificar_propio,
        muestra,
        "Sexo"
    )

    tiempo_pandas, _ = medir_tiempo(
        codificar_pandas,
        muestra,
        "Sexo"
    )

    memoria_propia, _ = medir_memoria(
        codificar_propio,
        muestra,
        "Sexo"
    )

    memoria_pandas, _ = medir_memoria(
        codificar_pandas,
        muestra,
        "Sexo"
    )

    resultados_codificacion.append({
        "n": n,
        "tiempo_propio_s": tiempo_propio,
        "tiempo_pandas_s": tiempo_pandas,
        "memoria_propia_kb": memoria_propia / 1024,
        "memoria_pandas_kb": memoria_pandas / 1024,
    })

resultados_codificacion = pd.DataFrame(
    resultados_codificacion
)

resultados_codificacion

,n,tiempo_propio_s,tiempo_pandas_s,memoria_propia_kb,memoria_pandas_kb
0,100,0.001050,0.001166,15.209961,14.458008
1,500,0.001118,0.001059,27.737305,29.295898
2,1000,0.000988,0.000986,45.458984,49.327148
3,2500,0.001014,0.000996,90.237305,93.295898
4,5000,0.001051,0.001129,173.458984,177.327148
5,5511,0.001038,0.001085,186.462891,188.890625


### 5.6 Análisis de eficiencia de la codificación

Las mediciones muestran un rendimiento similar entre `CodificadorOneHot` y `pandas.get_dummies`.

Las diferencias de tiempo son pequeñas y cambian según el tamaño evaluado. Para 5.511 observaciones, `CodificadorOneHot` registró aproximadamente 0,001038 s y `pandas.get_dummies` 0,001085 s. Esta diferencia es reducida, por lo que no resulta suficiente para establecer una ventaja relevante de rendimiento temporal entre ambas implementaciones.

El consumo de memoria también fue similar. Para 5.511 observaciones, `CodificadorOneHot` registró aproximadamente 186,46 KB de memoria pico y `pandas.get_dummies` 188,89 KB.

La complejidad temporal de la codificación puede expresarse como O(nk), donde `n` corresponde al número de observaciones y `k` al número de categorías generadas. La memoria necesaria para almacenar las nuevas columnas también crece como O(nk). Para `Sexo`, donde existen dos categorías fijas, el comportamiento respecto de `n` es aproximadamente lineal.

Considerando que ambas implementaciones producen resultados equivalentes y presentan un rendimiento similar, se mantiene `CodificadorOneHot` por razones arquitectónicas y de validación. La clase se integra directamente con la interfaz común `Transformador`, valida explícitamente los códigos categóricos y genera nombres semánticos consistentes con las categorías utilizadas en el proyecto.

### 5.7 Decisión técnica

Las pruebas realizadas permiten mantener las implementaciones `EscaladorEstandar` y `CodificadorOneHot` dentro del pipeline orientado a objetos.

En el escalamiento, la implementación propia produjo resultados numéricamente equivalentes a `StandardScaler` y presentó menores tiempos de ejecución en todos los tamaños evaluados. `StandardScaler`, sin embargo, presentó un menor consumo de memoria para los tamaños mayores. Ambas alternativas presentan un costo temporal lineal O(n).

En la codificación One-Hot, la implementación propia y `pandas.get_dummies` presentaron resultados equivalentes y un comportamiento temporal y espacial similar. Su complejidad general es O(nk), donde `k` representa el número de categorías.

La decisión de mantener las implementaciones propias no se fundamenta únicamente en diferencias de rendimiento. Estas clases forman parte de una arquitectura común basada en `Transformador`, separan responsabilidades, permiten validar los datos de entrada y mantienen la semántica de las variables de la ENS.

Por lo tanto, para el volumen actual del conjunto de datos, se prioriza la integración arquitectónica, la validación explícita y la mantenibilidad del pipeline, manteniendo documentados los compromisos observados entre tiempo y memoria.

> **Nota sobre la medición de memoria:** `tracemalloc` registra asignaciones de memoria administradas por Python. Por esta razón, los valores obtenidos se utilizan como evidencia comparativa entre implementaciones dentro del mismo entorno de ejecución y no como una medición absoluta de toda la memoria utilizada por el proceso.

## 6. Patrón de diseño Strategy aplicado a la imputación

Para `as27` había varias formas razonables de rellenar los nulos. Con
Strategy, cada forma es una clase con los mismos dos métodos
(`calcular` y `rellenar`), e `ImputadorFlexible` la recibe como
parámetro. Así se pueden comparar alternativas cambiando solo el
objeto que se entrega. Se comparan tres sobre `as27`: media, mediana
y mediana por tramo de `as28`.
P

In [10]:
estrategias_as27 = [PorMedia(), PorMediana(), PorMedianaDeTramo("as28")]
comparacion = comparar_estrategias(sin_hta, "as27", estrategias_as27)
comparacion

,estrategia,n_validos,nulos_restantes,media,desv_est,asimetria,cambio_desv_%
0,sin imputar,4518,993,469638.5,624613.8,11.31,0.00
1,media,5511,0,469638.5,565536.7,12.49,-9.46
2,mediana,5511,0,439072.1,569283.3,12.40,-8.86
3,mediana por tramo,4695,816,468979.0,620243.1,11.15,-0.70


La media y la mediana rellenan los 993 nulos y achican la dispersión
alrededor de un 9 %, porque llenan también a las 816 personas que no
declararon ingreso. La mediana por tramo casi no la altera (-0,70 %),
pero solo rellena 177: las otras 816 quedan sin valor.

La comparación no es de igual contra igual. Se elige la mediana por
tramo porque usa un dato que la persona sí entregó (su tramo) y evita
inventar un ingreso para quienes no respondieron nada. Su límite es que
esa no respuesta podría no ser aleatoria, y eso sigue como limitación
para la interpretación de los resultados.

La comparación anterior no requirió modificar `ImputadorFlexible`. La
celda siguiente lo muestra de forma directa: la misma clase se usa con
las tres estrategias y solo cambia el objeto que recibe.

In [11]:
for estrategia in estrategias_as27:
    paso = ImputadorFlexible("as27", estrategia)
    resultado = paso.ajustar_transformar(sin_hta)
    print(f"{estrategia.etiqueta:<20} nulos restantes en as27: {int(resultado['as27'].isna().sum())}")

media                nulos restantes en as27: 0
mediana              nulos restantes en as27: 0
mediana por tramo    nulos restantes en as27: 816


**Qué habría pasado sin el patrón.** En la Fase 2, cada alternativa
para `as27` significó una función distinta o una rama `if` dentro de
`imputar_nulos_numericos`. Sumar la mediana por tramo habría obligado
a modificar esa función y el código que la llama, con el riesgo de
alterar la imputación de otras columnas.

**Por qué Strategy y no otro patrón.** El problema de esta parte es
tener varias formas intercambiables de hacer lo mismo sobre una
columna. Factory, Observer y Singleton resuelven problemas distintos
(decidir qué objeto construir, registrar eventos o compartir una única
configuración) que no aparecen en esta etapa del proyecto.